# Case Study: Geographic Prescribing Map

**Prerequisite:** Run `00_build_database.ipynb` and `02_case_study/01_drug_selection.ipynb` first.

This notebook demonstrates the choropleth mapping methodology used to visualize state-level per-capita prescribing rates. Lenalidomide is used as the worked example.

The Streamlit app (`app.py`) generalizes this pipeline to any drug in the dataset with a simple dropdown interface. This notebook documents the underlying implementation.

In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: Build Per-Capita Table

Join lenalidomide state-level claims to Part D enrollment. Brand and generic rows are aggregated to one row per state. The FIPS code is used as the join key between the prescribing and enrollment tables.

In [ ]:
df = pd.read_sql_query("""
    SELECT
        p.Prscrbr_Geo_Desc AS state,
        e.BENE_STATE_ABRVTN AS state_abbr,
        SUM(p.Tot_Clms) AS tot_clms,
        SUM(p.Tot_Benes) AS tot_benes,
        ROUND(SUM(p.Tot_Drug_Cst), 0) AS tot_cost,
        e.PRSCRPTN_DRUG_TOT_BENES AS part_d_enrollees,
        ROUND(SUM(p.Tot_Clms) * 100000.0 / e.PRSCRPTN_DRUG_TOT_BENES, 1) AS clms_per_100k,
        ROUND(SUM(p.Tot_Drug_Cst) / e.PRSCRPTN_DRUG_TOT_BENES, 2) AS cost_per_enrollee
    FROM part_d_geo p
    JOIN state_enrollment e ON p.Prscrbr_Geo_Cd = e.BENE_FIPS_CD
    WHERE p.Prscrbr_Geo_Lvl = 'State'
      AND LOWER(p.Gnrc_Name) LIKE '%lenalidomide%'
    GROUP BY p.Prscrbr_Geo_Desc, e.BENE_STATE_ABRVTN, e.PRSCRPTN_DRUG_TOT_BENES
    ORDER BY clms_per_100k DESC
""", conn)

print(f"States in dataset: {len(df)}")
df.head(10)

## Step 2: Choropleth Map — Claims per 100k Enrollees

Plotly's `USA-states` locationmode requires two-letter state abbreviations. The color scale range is set dynamically to 105% of the maximum value to avoid overcompression of the color gradient.

In [ ]:
max_val = df['clms_per_100k'].max()
color_range = [0, round(max_val * 1.05, 1)]

fig = px.choropleth(
    df,
    locations='state_abbr',
    locationmode='USA-states',
    color='clms_per_100k',
    scope='usa',
    color_continuous_scale='Blues',
    range_color=color_range,
    title='Lenalidomide Claims per 100,000 Medicare Part D Enrollees by State (2023)',
    hover_name='state',
    hover_data={
        'state_abbr': False,
        'tot_clms': True,
        'tot_benes': True,
        'clms_per_100k': True
    },
    labels={'clms_per_100k': 'Claims per 100k'}
)

fig.update_layout(coloraxis_colorbar=dict(title='Claims per 100k'), height=550)
fig.show()
fig.write_html(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\notebooks\02_case_study\lenalidomide_map.html")

## Step 3: Cost per Enrollee Map

A second map showing Medicare drug spend per Part D enrollee by state. This metric captures both prescribing volume and drug cost variation, and is more relevant for healthcare cost surveillance than raw claim counts.

In [ ]:
max_cost = df['cost_per_enrollee'].max()
cost_range = [0, round(max_cost * 1.05, 2)]

fig2 = px.choropleth(
    df,
    locations='state_abbr',
    locationmode='USA-states',
    color='cost_per_enrollee',
    scope='usa',
    color_continuous_scale='Reds',
    range_color=cost_range,
    title='Lenalidomide Medicare Spend per Part D Enrollee by State (2023)',
    hover_name='state',
    hover_data={
        'state_abbr': False,
        'tot_cost': True,
        'cost_per_enrollee': True
    },
    labels={'cost_per_enrollee': 'Cost per Enrollee ($)'}
)

fig2.update_layout(coloraxis_colorbar=dict(title='Cost per Enrollee ($)'), height=550)
fig2.show()